In [6]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from PIL import Image
import io

# Import your backend logic
import model 

In [7]:
# ---------------- CONFIGURATION ----------------
# Path to your image folder
DATASET_FOLDER = "./../../dataset/basket_unique" 

In [8]:
# Ensure models are loaded
if 'models' not in locals():
    print("Loading models...")
    models = model.load_core_models()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading models...
Loading models on cpu...


Some weights of T5ForSequenceClassification were not initialized from the model checkpoint at t5-base and are newly initialized: ['classification_head.dense.bias', 'classification_head.dense.weight', 'classification_head.out_proj.bias', 'classification_head.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
# ---------------- HELPER: PARSE LABEL ----------------
def get_true_label(filename):
    """
    Parses sentiment from filename.
    Assumes format like: 'positive_123.jpg' or '123_negative.jpg'
    """
    filename_lower = filename.lower()
    if "positive" in filename_lower:
        return "Positive"
    elif "negative" in filename_lower:
        return "Negative"
    elif "neutral" in filename_lower:
        return "Neutral"
    return None

In [17]:
# ---------------- MAIN ANALYSIS LOOP ----------------
results = []
valid_extensions = ('.jpg', '.jpeg', '.png', '.JPG')
image_files = [f for f in os.listdir(DATASET_FOLDER) if f.lower().endswith(valid_extensions)]

print(f"Found {len(image_files)} images. Starting analysis...")

Found 2054 images. Starting analysis...


In [ ]:
for img_name in tqdm(image_files):
    img_path = os.path.join(DATASET_FOLDER, img_name)
    
    # 1. Get Ground Truth
    true_label = get_true_label(img_name)
    if not true_label:
        continue # Skip if no label found in filename
        
    # 2. Prepare Image
    try:
        image_pil = Image.open(img_path).convert("RGB")
        # specific to your pipeline: needs bytes for OCR check (if using API)
        # or filename lookup if relying on CSV
        with open(img_path, "rb") as f:
            image_bytes = f.read()
            
        # 3. Run MeViTSA Inference
        # Note: We pass img_name because your model looks it up in the CSV for OCR
        prediction = model.mevitsa_analysis(
            image_filename=img_name, 
            image_pil=image_pil, 
            image_bytes=image_bytes, 
            models_dict=models,
            gcv_api=0
        )
        
        # 4. Log Data
        results.append({
            "filename": img_name,
            "true_label": true_label,
            "pred_label": prediction['final_class'],
            "confidence": max(prediction['final_probs']),
            "text_source": prediction['text_source'],
            "alpha": prediction['alpha_used'],
            "extracted_text": prediction['text_content'][:50] + "..." # Truncate for display
        })
        
    except Exception as e:
        print(f"Error processing {img_name}: {e}")
    # print(results)
    # break

In [ ]:
# ---------------- METRICS & VISUALIZATION ----------------
df_results = pd.DataFrame(results)

# 1. Basic Accuracy
acc = accuracy_score(df_results['true_label'], df_results['pred_label'])
print(f"\n✅ Overall Accuracy: {acc:.2%}")

# 2. Classification Report
print("\n📊 Classification Report:")
print(classification_report(df_results['true_label'], df_results['pred_label']))

# 3. Confusion Matrix
plt.figure(figsize=(8, 6))
labels = ["Negative", "Neutral", "Positive"]
cm = confusion_matrix(df_results['true_label'], df_results['pred_label'], labels=labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title(f"MeViTSA Confusion Matrix (Acc: {acc:.2f})")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


# 4. Failure Analysis (High Confidence Errors)
print("\n⚠️ Top 5 High-Confidence Errors (Worst Failures):")
errors = df_results[df_results['true_label'] != df_results['pred_label']]
top_errors = errors.sort_values(by="confidence", ascending=False).head(5)

display(top_errors[['filename', 'true_label', 'pred_label', 'confidence', 'text_source', 'extracted_text']])